# 🫀 퀘스트 46 · Q7-U — **자를 빌려서 내 가설을 잰다: 공개 delineator × QRST 소거**

| | **MedKOS / `notebooks/quest46_q7u_public_delineator.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` |
| 앞선 실험 | **Q7-T**(BUT PDB 검증 · T1 ❌ · **T2 ✅**) · `ailab-2026-0067`(Q7-R) |
| 규약 | **R16 · R17 · R22 · R29 ① ② · R30 ① · R33 ① · R34 ① ③ ④ ⑤** |
| 학습 | **0회** — 신호처리만 · GPU 불필요 |

## Q7-T 가 남긴 것

```
T1 ❌  argmax|잔차| 피크 픽커     Se 0.4375  vs 벤치마크 0.9307  → 종결 조건 ① 발동
T2 ✅  QRST 소거가 P 를 드러낸다  순위 0.3070 → 0.5895 (+0.2825 [+0.2273, +0.3315])
T3 ❌  「P 부재」 AUROC 0.5190    ← **독립 증거가 아니다**(T1 이 실패한 검출기의 출력을 씀)
```

죽은 건 **검출기**이지 가설이 아니다. 검출기는 **자**다 — 자가 부정확하면 가설이 맞는지
틀린지를 잴 수가 없다. 그래서 자를 **빌린다**.

## ★★ 그런데 이게 이 런의 요점은 아니다

```
공개 방법이 하는 것    원 ECG          →  delineator  →  P
Q7-U 가 묻는 것        QRST 소거 잔차  →  delineator  →  P     ← 아무도 안 했다
```

phasor transform·wavelet delineator 는 **전부 원신호**에서 P 를 찾는다. 「P 가 QRS·T 에
묻히면 QRS·T 를 빼서 꺼낸다」는 발상은 그 앞단에 붙는 별개의 층이고, **T2 가 이미
그 층이 P 를 드러낸다는 걸 외부 정답으로 보였다.**

**주 관문 U2 는 그래서 짝지은 차다** — 같은 검출기, 입력만 바꿔서.

## 자 두 개 — 각각 뭘 하는가

**`dwt`** (NeuroKit2 `ecg_delineate(method="dwt")` · Martínez 2004 계열)
이차 스플라인 웨이블릿으로 dyadic scale 분해. QRS 는 scale 2¹~2³, **P·T 는 2⁴~2⁵** 에
산다 — scale 선택이 곧 그 파형에 맞춘 대역통과다. 정점은 **부호 반대인 modulus maximum
쌍 사이의 영교차**로 잡는다 → 진폭 문턱이 아니라 **모양**으로 잡으므로 작아도 살아남는다.

**`phasor`** (Martínez–Alcaraz–Rieta, Physiol Meas 31:1467, 2010)
P 가 안 잡히는 근본 이유는 **동적 범위**다(QRS 1mV vs P 0.1mV). 각 샘플을 위상으로 바꾼다.

```
y(n) = Rv + j·x(n)          φ(n) = arctan( x(n) / Rv )
```

`arctan` 은 0 근처에서 가파르고 큰 값에서 포화 → **작은 편향은 증폭, QRS 는 압축**.
`Rv` 가 무릎 위치를 정한다. ⚠️ **완전한 판본이 아니다** — 원 논문은 위상으로 onset/offset
까지 구획한다. 여기선 **P 정점만** 잡는 축소판이고 그렇게만 부른다.

★ 벤치마크 **Saclova et al., Sci Rep 12:6589 (2022)** 가 BUT PDB 에서 **Se 0.9307 / PP
0.8860** 인데, 그 방법이 바로 **phasor transform + 부정맥별 결정 규칙**이다. 즉 `phasor`
팔은 「벤치마크 계열을 재현할 수 있나」이기도 하다(결정 규칙 없이 얼마나 가나).

## ★ 프로토타입이 잡은 설계 결함 둘

**① 재구성 자체가 신호를 바꾼다.** 소거는 비트 단위인데 delineator 는 연속 신호를 받는다.
비트 잔차를 **R 최근접 Voronoi 분할**로 되붙여 연속 신호를 만드는데, 그 이음매가
파형이다. 그래서 소거 없이 **재구성만 한** `none` 팔을 넣는다 — 소거 효과와 재구성
비용을 가른다. (실측: 레코드 01 에서 raw Se 0.7293 → 재구성 `none` 0.6391. 이걸 안
갈랐으면 소거 탓으로 돌렸을 것이다.)

**② delineator 도 기권하지 않는다.** NeuroKit2 는 비트마다 P 후보를 하나 낸다. 실측에서
레코드 50(P 가 거의 없는 기록)은 **검출 215개 · PPV 0.0279** 였다. 벤치마크가 Se 0.93 을
내는 건 **부정맥별 결정 규칙으로 「여기엔 P 가 없다」를 판정**하기 때문이다. 그래서
Q7-T 에서 쓴 **LORO Youden 기권 문턱**을 그대로 얹는다(자기 레코드 정답은 안 본다).

## 사전등록 — 관문

| 관문 | 내용 | 판정 |
|---|---|---|
| **U1 ★★** | **자가 서는가** — LORO 선택 팔이 전문가 주석 대비 | **Se ≥ 0.70 AND PPV ≥ 0.70** · 벤치마크 0.9307/0.8860 병기 |
| **U2 ★★(주)** | **소거가 검출을 돕는가** — 같은 검출기, `abs`/`st` vs **`none`** 짝지은 차 | ΔSe·ΔPPV 의 CI 하한 > 0 |
| **U3 ★** | **재구성 비용** — `none` vs `raw` | 보고 · U2 의 해석 근거 |
| **U4 ★** | 시공간 보정이 검출에도 이득인가 — `st` vs `abs` | 짝지은 차 |
| **U5 ★** | **「P 부재」 재측정** — Q7-T 의 T3 을 **작동하는 검출기로** 다시 | AUROC CI 하한 > 0.65 |

### 판정표 (R29 ②)

- **U1 ✅ · U2 ✅** → ★★ **님 발상이 검증된 자로 확증**. 그리고 그건 벤치마크를
  **개선하는** 방향이다(그들은 원신호에서만 했다). → **Q7-S′** 로 간다
- **U1 ✅ · U2 ❌** → 자는 섰는데 소거는 검출에 안 보탠다. T2(순위)와 U2(검출)가
  갈리는 것이므로 **소거는 특징 생성에만** 쓰고 검출은 원신호에서 한다
- **U1 ❌** → 공개 방법으로도 못 세운다 → 이 규모에서 P 검출은 우리 문제가 아니다.
  **형태 갈래를 닫거나** 학습형 검출기로 간다(그건 외부 정답 성격을 잃는 대가를 치른다)
- **어느 것이든 ⛔ 측정 불가** → 어떤 결론 분기도 타지 않는다

⚠️ **이 런도 SVEB 질문에 답하지 않는다** — 「P 를 볼 수 있는가 · 소거가 그걸 돕는가」까지다.

In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def mde(lo, hi):
    """최소 검출 효과 = CI 반폭(R33 ①)."""
    return (hi - lo) / 2.0 if np.isfinite(lo) and np.isfinite(hi) else float("nan")

def need_n(n, lo, hi, mean, margin):
    if not (np.isfinite(lo) and np.isfinite(hi) and np.isfinite(mean)) or n < 1:
        return float("nan")
    slack = margin - abs(mean)
    return None if slack <= 0 else float(n) * (((hi - lo) / 2.0) / slack) ** 2

def boot_mean(v, seed, nb=3000, q=2.5):
    """레코드 단위 부트스트랩 평균 + CI."""
    d = np.asarray(v, float); d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), len(d)
    rng = np.random.RandomState(seed)
    b = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)]
    return (float(d.mean()), float(np.percentile(b, q)),
            float(np.percentile(b, 100 - q)), len(d))

def match_1d(det, ref, tol):
    """**탐욕적 1:1** 매칭(R34 ①). 안 걸면 검출 하나가 정답 여럿을 먹어 Se 가 부푼다."""
    det = np.sort(np.asarray(det, float)); ref = np.sort(np.asarray(ref, float))
    used = np.zeros(len(det), bool); errs = []
    for r in ref:
        cand = np.where((~used) & (np.abs(det - r) <= tol))[0]
        if len(cand):
            j = cand[np.argmin(np.abs(det[cand] - r))]
            used[j] = True; errs.append(float(det[j] - r))
    return len(errs), np.asarray(errs, float)

def se_ppv(det, ref, tol):
    """Se = 매칭/정답수 · PPV = 매칭/**발화수**. 정답 전수를 분모로 쓴다(벤치마크와 같게)."""
    m, e = match_1d(det, ref, tol)
    return m / max(len(ref), 1), m / max(len(det), 1), e

class AssetError(RuntimeError): pass
print("CELL 0 ✅")

In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, importlib, time, warnings
importlib.invalidate_caches()
warnings.filterwarnings("ignore")
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0 = 20260804

# ── 좌표계: **레코드의 원 샘플레이트에서 그대로** 판정한다.
#    Q7-T 는 360Hz 로 리샘플했지만(SVDB 정합용) 여기선 SVDB 를 안 건드리므로
#    리샘플 인공물을 들일 이유가 없다. 창은 **밀리초**로 정의해 fs 마다 환산한다.
BEAT_PRE_MS, BEAT_POST_MS = 278.0, 556.0     # 비트 절단(소거용)
FIT_LO_MS, FIT_HI_MS = -42.0, 417.0          # ★ 적합은 **심실 구간에서만**
SHIFT_MS = 11.0                              # 시공간-lite 의 비트별 시프트 탐색 폭

# ── ★★ 팔: 검출기 2 × 입력 4
DETECTORS = ("dwt", "phasor")
INPUTS = ("raw", "none", "abs", "st")        # ★ `none` = **재구성만**(소거 없음) — 대조
ARMS = tuple(f"{d}|{i}" for d in DETECTORS for i in INPUTS)

# ── phasor transform (Martinez-Alcaraz-Rieta 2010) — 축소판
RV_FRACS = (0.25, 0.5, 1.0, 2.0)             # Rv = frac x MAD(신호) · **LORO 로 고른다**
P_LO_MS, P_HI_MS = -278.0, -42.0             # P 탐색창(R 기준)

# ── ★★ 관문 사전등록
TOL_MS = 50.0                                # P 위치 허용 오차(문헌 관행)
SE_MIN, PPV_MIN = 0.70, 0.70
BENCH_SE, BENCH_PP = 0.9307, 0.8860          # Saclova et al. 2022, BUT PDB
U5_AUROC_MIN = 0.65
BUT_DIR = "but-pdb/1.0.0"
MIN_P = 5                                    # R17 — 정답이 이보다 적은 레코드는 판정에서 뺀다

RULE_CHECK = {
    "R16 fallback 없음":     "BUT PDB·neurokit2 실패 시 **중단** — 합성으로 대체하지 않는다",
    "R17 최소 n":            f"정답 P 가 {MIN_P} 미만인 레코드는 판정에서 뺀다",
    "R22 누수 없음":         "검출기는 정답을 **인자로도 안 받는다** · 기권 문턱·팔 선택은 "
                             "**다른 레코드**에서만(LORO) · 채점만 평가 대상 정답으로",
    "R29 ② 측정 불가 분기 금지": "⛔ 판정은 어떤 결론 분기도 타지 않는다",
    "R30 ① 필요표본":        "미결이면 필요 레코드 수를 계산해 출력",
    "R33 ① MDE":             "관문마다 MDE(CI 반폭)를 내고 점추정과 비교",
    "R34 ① 판정 규칙 명시":  "★ 매칭은 **탐욕적 1:1** · Se 분모는 **정답 전수**",
    "R34 ③ 대조 보장":       "★ `none`(재구성만)이 **구성으로 보장된 대조** — 소거 효과와 "
                             "재구성 비용을 가른다",
    "R34 ④ 문턱 근거":       "★ TOL ±50ms 는 문헌 관행 · Rv 는 격자에서 **LORO 로** 고른다",
    "R34 ⑤ 종결 조건":       "U1 이 공개 방법으로도 실패하면 형태 갈래를 닫는다",
}

CONFIG = dict(
    exp="quest46_q7u_public_delineator", quest="ailab-2026-0046", step="public-delineator",
    parent_exp=["quest46_q7t_p_anchored"],
    purpose=("Q7-T 는 T1(자체 검출기) ❌ · **T2(소거가 P 를 드러낸다) ✅** 로 끝났다. "
             "죽은 건 **검출기**이지 가설이 아니다 — 검출기는 자다. 그래서 자를 "
             "**빌린다**(NeuroKit2 DWT · phasor transform). 그런데 공개 방법은 **전부 "
             "원신호**에서 P 를 찾는다. **주 관문 U2 는 같은 검출기를 소거 잔차에 걸면 "
             "원신호보다 나은가**이고, 그게 이 퀘스트 고유의 질문이다. 아무도 안 한 조합"),
    dataset="BUT PDB(50x2분x2유도 · 전문가 2명 P 주석)",
    detectors=list(DETECTORS), inputs=list(INPUTS), arms=list(ARMS),
    beat_ms=[BEAT_PRE_MS, BEAT_POST_MS], fit_ms=[FIT_LO_MS, FIT_HI_MS],
    p_win_ms=[P_LO_MS, P_HI_MS], rv_fracs=list(RV_FRACS), tol_ms=TOL_MS,
    se_min=SE_MIN, ppv_min=PPV_MIN, min_p=MIN_P,
    benchmark=dict(paper="Saclova et al., Sci Rep 12:6589 (2022)", se=BENCH_SE, pp=BENCH_PP,
                   method="phasor transform + 부정맥별 결정 규칙", db="BUT PDB"),
    method_refs=["Martinez, Almeida, Olmos, Rocha, Laguna, IEEE TBME 51:570-581 (2004) — "
                 "wavelet delineator (NeuroKit2 `dwt`)",
                 "Martinez, Alcaraz, Rieta, Physiol Meas 31:1467 (2010) — phasor transform",
                 "Stridh & Sornmo, IEEE TBME 48:105-111 (2001) — QRST cancellation"],
    rule_check=RULE_CHECK,
    predictions={
        "U1": f"★★ **자가 서는가** — LORO 선택 팔이 Se ≥ {SE_MIN} AND PPV ≥ {PPV_MIN}. "
              f"벤치마크 Se {BENCH_SE:.4f}/PP {BENCH_PP:.4f} 병기. Q7-T 자체 검출기는 "
              "Se 0.4375 였다",
        "U2": "★★ **(주 관문) 소거가 검출을 돕는가** — 같은 검출기로 `abs`/`st` 대 "
              "**`none`**(재구성만)의 **짝지은** ΔSe·ΔPPV. CI 하한 > 0. ★ 대조를 `raw` 가 "
              "아니라 `none` 으로 잡는 게 핵심이다 — 재구성 인공물을 소거 탓으로 돌리지 "
              "않기 위해서(프로토타입 실측: 레코드 01 raw 0.7293 → none 0.6391)",
        "U3": "★ **재구성 비용** — `none` vs `raw`. U2 의 해석 근거이자, 잔차 경로를 "
              "쓸 때 치르는 값을 명시",
        "U4": "★ 시공간 보정이 **검출에도** 이득인가 — `st` vs `abs` 짝지은 차. "
              "Q7-T 에서 심실 잔차 RMS 는 st 가 abs 의 절반이었다(0.1304 vs 0.2540)",
        "U5": f"★ **「P 부재」 재측정** — Q7-T 의 T3(AUROC 0.5190 ❌)은 실패한 검출기의 "
              f"출력을 써서 **독립 증거가 아니었다**. 작동하는 검출기로 다시 잰다. "
              f"AUROC CI 하한 > {U5_AUROC_MIN}"},
    caveat=("★ **`phasor` 는 원 논문의 완전한 판본이 아니다** — 원 논문은 위상으로 "
            "onset/offset 까지 구획한다. 여기선 **P 정점만** 잡는 축소판이다. "
            "★ **`st` 도 Stridh-Sornmo 의 2유도 축소판**이다(Q7-T 와 같음). "
            "★ **NeuroKit2 는 기권하지 않는다** — 비트마다 후보를 하나 낸다. 벤치마크가 "
            "Se 0.93 을 내는 건 **부정맥별 결정 규칙으로 P 부재를 판정**하기 때문이다. "
            "그래서 **LORO Youden 기권 문턱**을 얹는다(실측: 레코드 50 은 기권 없이 "
            "검출 215개 · PPV 0.0279 였다). "
            "★ **R 위치는 `qrs` 주석을 쓴다** — P 정답이 아니므로 R22 위반이 아니고, "
            "이 퀘스트 내내 비트 정렬 기준으로 써 온 값이다. "
            "★ **이 런도 SVEB 질문에 답하지 않는다.** 학습 0회 · 예상 20~35분"))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q7u_public_delineator", CONFIG, project=PROJECT)
run.log("설정 ✅ **자를 빌려서 내 가설을 잰다** — 주 관문은 U2(소거 잔차 vs 재구성 대조)")
run.log(f"  팔 {len(ARMS)}개 = 검출기 {DETECTORS} x 입력 {INPUTS}")
run.log(f"  ★ `none` 은 **재구성만 하고 소거는 안 한** 대조다 — 재구성 비용을 소거 탓으로")
run.log("    돌리지 않기 위해서(R34 ③ 구성 보장 대조)")
run.log(f"  허용 오차 ±{TOL_MS:.0f}ms · 매칭 **탐욕적 1:1** · Se 분모 = **정답 전수**")
run.log(f"  벤치마크 — Saclova 2022 BUT PDB: Se {BENCH_SE:.4f} / PP {BENCH_PP:.4f}")
run.log("            (그 방법이 **phasor transform + 부정맥별 결정 규칙**이다)")
run.log("\n  사전등록 규칙 체크리스트 (R29 ③)")
for k_, v_ in RULE_CHECK.items():
    run.log(f"    [x] {k_:<22} {v_}")

In [ ]:
# CELL 2 — 【U-0a】 자산 적재 — BUT PDB · neurokit2 (fallback 없음 R16)
try:
    import wfdb
except ModuleNotFoundError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wfdb"], check=True)
    importlib.invalidate_caches(); import wfdb
try:
    import neurokit2 as nk
except ModuleNotFoundError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "neurokit2"], check=True)
    importlib.invalidate_caches(); import neurokit2 as nk
import re, urllib.request

run.log("\n" + "=" * 100)
run.log("【U-0a】 BUT PDB — 전문가 P 주석 (외부 정답)")
run.log("=" * 100)
run.log(f"  neurokit2 {nk.__version__} · wfdb {getattr(wfdb, '__version__', '?')}")

BASE = f"https://physionet.org/files/{BUT_DIR}"
BEAT_SYM = set("NLRAaJSVFejE/fQ")

def _get(url, timeout=60):
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return r.read().decode("utf-8", "replace")

BUT_RECS = [str(r) for r in wfdb.get_record_list("but-pdb")]     # ⛔ 실패 시 중단(R16)
if len(BUT_RECS) < 10:
    raise AssetError(f"BUT PDB 레코드 목록이 {len(BUT_RECS)}개 — 다운로드 실패")

# ★ Q7-T 가 여기서 두 번 멈췄다. RECORDS 이름과 실제 파일명이 다르고(`1` vs `01.hea`),
#   주석 확장자도 추측하면 틀린다. 둘 다 **실물로 확인**한다.
def resolve_rid(r):
    for c in dict.fromkeys([r] + ([f"{int(r):0{w}d}" for w in (2, 3)] if r.isdigit() else [])):
        try:
            wfdb.rdheader(c, pn_dir=BUT_DIR); return c
        except Exception:
            continue
    return None

_res = [(r, resolve_rid(r)) for r in BUT_RECS]
BUT_RECS = [c for _, c in _res if c is not None]
_pad = sum(1 for r, c in _res if c is not None and c != r)
if len(BUT_RECS) < 10:
    raise AssetError(f"헤더가 읽히는 레코드가 {len(BUT_RECS)}개")
run.log(f"  레코드 {len(BUT_RECS)}/{len(_res)}"
        + (f" · **{_pad}개는 0 채움 필요**(RECORDS 와 파일명 불일치)" if _pad else ""))

def list_exts(rid):
    exts = []
    try:
        for ln in _get(f"{BASE}/ANNOTATORS").splitlines():
            tok = ln.split("\t")[0].strip() if ln.strip() else ""
            if tok and not tok.startswith("#"):
                exts.append(tok.split()[0])
    except Exception as e:
        run.log(f"  ⚠️ ANNOTATORS 를 못 읽었다: {type(e).__name__} {e}")
    if not exts:
        try:
            html = _get(BASE + "/")
            exts = sorted({x for x in re.findall(rf"{re.escape(rid)}\.([A-Za-z0-9_]+)", html)
                           if x not in ("dat", "hea", "xws", "png", "txt")})
        except Exception as e:
            run.log(f"  ⚠️ 디렉터리 목록도 못 읽었다: {type(e).__name__} {e}")
    return exts

PROBE = []
for e in list_exts(BUT_RECS[0]):
    try:
        a = wfdb.rdann(BUT_RECS[0], e, pn_dir=BUT_DIR)
        PROBE.append((e, len(a.sample), sorted(set(a.symbol))))
    except Exception:
        pass
if not PROBE:
    raise AssetError(f"{BUT_RECS[0]}: 읽히는 주석 파일이 하나도 없다")
EXT_P = next((e for e, _, _ in PROBE if e.lower().startswith("p")), None)
_rest = [(e, n_, sy) for e, n_, sy in PROBE if e != EXT_P]
EXT_Q = max(_rest, key=lambda t: (len(set(t[2]) & BEAT_SYM), t[1]))[0] if _rest else None
if EXT_P is None or EXT_Q is None:
    raise AssetError(f"주석 역할을 못 가렸다 — {[(e, n) for e, n, _ in PROBE]}")
run.log(f"  주석 확장자 — QRS `{EXT_Q}` · P `{EXT_P}` (**알아낸 값**)")

BUT, T0 = {}, time.time()
for rid in BUT_RECS:
    rec = wfdb.rdrecord(rid, pn_dir=BUT_DIR)
    sig = np.nan_to_num(np.asarray(rec.p_signal, float), nan=0.0, posinf=0.0, neginf=0.0)
    if sig.ndim != 2 or sig.shape[1] < 2:
        raise AssetError(f"{rid}: 2유도가 아니다 {sig.shape}")
    rp = np.asarray(wfdb.rdann(rid, EXT_Q, pn_dir=BUT_DIR).sample, int)
    pp = np.asarray(wfdb.rdann(rid, EXT_P, pn_dir=BUT_DIR).sample, int)
    if len(rp) < 5 or len(pp) < MIN_P:
        continue                                          # R17
    BUT[rid] = dict(sig=sig[:, :2], fs=int(rec.fs), r=rp, p=pp)
run.log(f"  적재 {len(BUT)}개 · {time.time()-T0:.0f}초")
_fs = sorted({v["fs"] for v in BUT.values()})
_nq = sum(len(v["r"]) for v in BUT.values()); _np_ = sum(len(v["p"]) for v in BUT.values())
run.log(f"  샘플레이트 {_fs} — **리샘플하지 않는다**(각 레코드 좌표에서 판정 · 창은 ms 정의)")
run.log(f"  QRS **{_nq:,}** · 주석 P **{_np_:,}** · P/QRS {_np_/max(_nq,1):.3f}")
run.log(f"  (문헌: QRS 7,638 · P 5,437 · P 없는 QRS 2,201(28.8%))")
CONFIG["but"] = dict(n_rec=len(BUT), n_qrs=int(_nq), n_p=int(_np_),
                     fs=[int(f) for f in _fs], ann_ext=dict(qrs=EXT_Q, pwave=EXT_P),
                     pad_fixed=int(_pad))
run.save_json("config", CONFIG)

In [ ]:
# CELL 3 — 【U-A】 QRST 소거 → **연속 신호로 재구성**
# Stridh & Sornmo (2001). ⚠️ `st` 는 2유도 축소판이다(Q7-T 와 같음).
# ★ delineator 는 연속 신호를 받는데 소거는 비트 단위다. 되붙이는 규칙이 필요하고,
#   그 규칙 자체가 파형을 바꾼다 → 그래서 `none`(재구성만) 대조가 있다.
def ms2s(ms, fs):
    return int(round(ms * fs / 1000.0))

def cancel_full(sig, rp, fs, mode):
    """비트별 QRST 소거 후 **R 최근접 Voronoi 분할**로 연속 신호 재구성(겹침·틈 없음).

    ★ 적합은 **심실 구간에서만** — P 구간이 적합에 끼면 P 를 같이 지운다.
    ★ `mode="none"` 은 **소거를 안 하고 재구성만** 한다 — 재구성 비용을 재는 대조."""
    pre, post = ms2s(BEAT_PRE_MS, fs), ms2s(BEAT_POST_MS, fs)
    L = pre + post
    keep = (rp >= pre) & (rp < len(sig) - post)
    rk = rp[keep]
    if len(rk) < 3:
        raise AssetError("소거할 비트가 3 미만")
    B = np.stack([sig[p - pre:p - pre + L, :2].T for p in rk]).astype(float)
    if mode == "none":
        R_ = B.copy()
    else:
        T = np.median(B, axis=0)
        if mode == "abs":
            R_ = B - T[None]
        elif mode == "st":
            lo, hi = pre + ms2s(FIT_LO_MS, fs), pre + ms2s(FIT_HI_MS, fs)
            lo, hi = max(lo, 0), min(hi, L)
            fit = slice(lo, hi)
            sh = ms2s(SHIFT_MS, fs)
            R_ = B - T[None]; best = np.full(len(B), np.inf)
            for d in range(-sh, sh + 1):
                Ts = np.stack([np.roll(T[j], d) for j in range(2)])
                X = np.c_[Ts[0, fit], Ts[1, fit], np.ones(hi - lo)]
                Pinv = np.linalg.pinv(X); Xf = np.c_[Ts[0], Ts[1], np.ones(L)]
                r0 = B[:, 0, :] - (Xf @ (Pinv @ B[:, 0, fit].T)).T
                r1 = B[:, 1, :] - (Xf @ (Pinv @ B[:, 1, fit].T)).T
                err = (r0[:, fit] ** 2).sum(1) + (r1[:, fit] ** 2).sum(1)
                im = err < best
                if im.any():
                    best[im] = err[im]; R_[im, 0, :] = r0[im]; R_[im, 1, :] = r1[im]
        else:
            raise AssetError(f"소거 방식 {mode} 를 모른다")
    out = np.zeros_like(sig[:, :2])
    mid = np.r_[0, (rk[:-1] + rk[1:]) // 2, len(sig)]
    for i, p in enumerate(rk):
        a, b = max(mid[i], p - pre), min(mid[i + 1], p - pre + L)
        if b > a:
            out[a:b, :] = R_[i, :, a - (p - pre):b - (p - pre)].T
    return out

run.log("\n" + "=" * 100)
run.log("【U-A】 QRST 소거 + 연속 재구성 — " + " · ".join(INPUTS))
run.log("=" * 100)
SIGS, T1_ = {}, time.time()
for rid, d in BUT.items():
    SIGS[rid] = {"raw": d["sig"]}
    for m in ("none", "abs", "st"):
        SIGS[rid][m] = cancel_full(d["sig"], d["r"], d["fs"], m)
run.log(f"  ({time.time()-T1_:.0f}초) 입력별 유도0 RMS")
for m in INPUTS:
    run.log(f"    {m:<5} {np.mean([float(np.sqrt((SIGS[r][m][:, 0] ** 2).mean())) for r in BUT]):.5f}")
run.log("  ▸ `none` 은 재구성만 한 것이라 `raw` 와 RMS 가 비슷해야 정상이다")
run.log("  ▸ `abs`·`st` 는 심실이 빠져 작아져야 한다 · `st` < `abs` 면 시공간 보정 작동")
CONFIG["input_rms"] = {m: float(np.mean(
    [np.sqrt((SIGS[r][m][:, 0] ** 2).mean()) for r in BUT])) for m in INPUTS}
run.save_json("config", CONFIG)

In [ ]:
# CELL 4 — 【U-B】 검출기 — `dwt`(NeuroKit2) · `phasor`(Martinez 2010 축소판)
# ★ 두 검출기 모두 **정답을 인자로도 받지 않는다**(R22). R 위치만 받는다 —
#   R 은 P 정답이 아니고 이 퀘스트 내내 비트 정렬 기준으로 써 온 값이다.
from scipy.signal import medfilt

def _detrend_win(v):
    """창 안 선형 추세 제거 — 기저동요가 정점을 밀지 않게."""
    n = len(v)
    if n < 3:
        return v - np.mean(v) if n else v
    t = np.arange(n, dtype=float)
    A = np.c_[t, np.ones(n)]
    return v - A @ np.linalg.lstsq(A, v, rcond=None)[0]

def detect_dwt(x, rp, fs):
    """NeuroKit2 이산 웨이블릿 구획(Martinez 2004 계열).
    QRS 는 scale 2^1~2^3 · P/T 는 2^4~2^5 → scale 선택이 곧 파형별 대역통과.
    정점은 부호 반대인 modulus maximum 쌍의 **영교차**로 잡는다(진폭 문턱이 아니다)."""
    _, w = nk.ecg_delineate(x, rpeaks=rp, sampling_rate=fs, method="dwt")
    p = w.get("ECG_P_Peaks", [])
    return np.asarray(sorted({int(v) for v in p
                              if v is not None and np.isfinite(v) and 0 <= v < len(x)}), int)

def detect_phasor(x, rp, fs, rv_frac):
    """phasor transform (Martinez-Alcaraz-Rieta, Physiol Meas 31:1467, 2010) — **축소판**.

        y(n) = Rv + j*x(n)      phi(n) = arctan( x(n) / Rv )

    arctan 은 0 근처에서 가파르고 큰 값에서 포화 → **작은 편향(P)은 증폭, QRS 는 압축**.
    동적 범위 문제(QRS 1mV vs P 0.1mV)를 비선형으로 누르는 게 요점이다.
    ⚠️ 원 논문은 위상으로 onset/offset 까지 구획한다 — 여기선 **P 정점만** 잡는다."""
    mad = float(np.median(np.abs(x - np.median(x)))) + 1e-12
    phi = np.arctan(x / (rv_frac * mad))
    lo, hi = ms2s(P_LO_MS, fs), ms2s(P_HI_MS, fs)          # 둘 다 음수(R 앞)
    out = []
    for p in rp:
        a, b = p + lo, p + hi
        if a < 0 or b > len(phi) or b - a < 3:
            continue
        seg = _detrend_win(phi[a:b])
        out.append(int(a + np.argmax(np.abs(seg))))
    return np.asarray(sorted(set(out)), int)

def score_at(x, pos, fs):
    """검출 위치의 **국소 두드러짐** = |추세제거 진폭| / 창 안 MAD. 기권 판정용 점수.
    ★ 정답을 안 본다. 검출기마다 다른 눈금을 쓰지 않도록 **같은 자**로 잰다."""
    lo, hi = ms2s(P_LO_MS, fs), ms2s(P_HI_MS, fs)
    out = []
    for q in pos:
        a, b = max(q + lo, 0), min(q + hi, len(x))
        if b - a < 3:
            out.append(0.0); continue
        seg = _detrend_win(x[a:b])
        m = float(np.median(np.abs(seg - np.median(seg)))) + 1e-12
        out.append(float(abs(seg[min(max(q - a, 0), len(seg) - 1)]) / m))
    return np.asarray(out, float)

run.log("\n" + "=" * 100)
run.log("【U-B】 검출 — " + " x ".join((str(DETECTORS), str(INPUTS))))
run.log("=" * 100)
T2_ = time.time()
DET = {}                       # (det, inp) -> rid -> dict(pos, sc)  · phasor 는 rv_frac 별
for rid, d in BUT.items():
    fs, rp = d["fs"], d["r"]
    for inp in INPUTS:
        x = SIGS[rid][inp][:, 0]
        try:
            DET.setdefault(("dwt", inp), {})[rid] = dict(
                pos=(pp := detect_dwt(x, rp, fs)), sc=score_at(x, pp, fs))
        except Exception as e:
            run.log(f"    ⚠️ dwt|{inp}|{rid} 실패: {type(e).__name__}")
        for rv in RV_FRACS:
            DET.setdefault(("phasor", inp, rv), {})[rid] = dict(
                pos=(pp := detect_phasor(x, rp, fs, rv)), sc=score_at(x, pp, fs))
run.log(f"  ({time.time()-T2_:.0f}초) 검출 완료 · 조합 {len(DET)}개")
for inp in INPUTS:
    n_ = np.mean([len(DET[("dwt", inp)][r]["pos"]) for r in DET.get(("dwt", inp), {})] or [np.nan])
    run.log(f"    dwt|{inp:<5} 레코드당 검출 {n_:.1f}개")
run.log("  ▸ 검출기는 **기권하지 않는다** — 비트마다 후보를 하나 낸다. 다음 셀에서")
run.log("    **LORO Youden 기권 문턱**을 얹는다(실측: 기권 없으면 P 희소 레코드 PPV 0.03)")

In [ ]:
# CELL 5 — 【U-C】 관문 U1 — 자가 서는가 (LORO 기권 + LORO 팔 선택)
from sklearn.metrics import roc_curve, roc_auc_score
run.log("\n" + "=" * 100)
run.log(f"【U-C】 U1 — 공개 delineator vs 전문가 주석 (±{TOL_MS:.0f}ms · 탐욕적 1:1)")
run.log("=" * 100)
VERD, DIFF = {}, {}
def g_(k, v, d):
    VERD[k] = v; run.log(f"  {k:<4}{v}  {d}")

RIDS = sorted(BUT)

def hit_labels(key, rid):
    """이 레코드의 검출 각각이 **정답과 맞았는지**(기권 문턱 학습용 라벨).
    ★ 이 라벨은 **다른 레코드의 문턱을 정할 때만** 쓴다 — 자기 레코드엔 안 쓴다(R22)."""
    d = BUT[rid]; tol = TOL_MS * d["fs"] / 1000.0
    pos = DET[key][rid]["pos"]
    if not len(pos):
        return np.zeros(0, int)
    ref = np.sort(d["p"].astype(float)); det = np.sort(pos.astype(float))
    used = np.zeros(len(det), bool)
    for r in ref:
        c = np.where((~used) & (np.abs(det - r) <= tol))[0]
        if len(c):
            used[c[np.argmin(np.abs(det[c] - r))]] = True
    order = np.argsort(pos)
    lab = np.zeros(len(pos), int); lab[order] = used.astype(int)
    return lab

def loro_thr(key, rid_out):
    """`rid_out` **을 뺀** 레코드들에서 Youden 최적 기권 문턱(R22)."""
    sc, y = [], []
    for r in RIDS:
        if r == rid_out or r not in DET.get(key, {}):
            continue
        sc.append(DET[key][r]["sc"]); y.append(hit_labels(key, r))
    if not sc:
        return float("-inf")
    sc = np.concatenate(sc); y = np.concatenate(y)
    if y.sum() < 5 or (1 - y).sum() < 5:
        return float("-inf")
    fpr, tpr, th = roc_curve(y, sc)
    return float(th[int(np.argmax(tpr - fpr))])

def evaluate(key):
    """레코드별 (Se, PPV, 발화율, 오차ms). 기권 문턱은 **레코드마다 LORO**."""
    out = {}
    for rid in RIDS:
        if rid not in DET.get(key, {}):
            continue
        d = BUT[rid]; tol = TOL_MS * d["fs"] / 1000.0
        pos, sc = DET[key][rid]["pos"], DET[key][rid]["sc"]
        fire = pos[sc >= loro_thr(key, rid)] if len(pos) else pos
        se, pp, err = se_ppv(fire, d["p"], tol)
        out[rid] = dict(se=se, pp=pp, fire=len(fire) / max(len(pos), 1),
                        err=(err / d["fs"] * 1000.0).tolist())
    return out

T3_ = time.time()
# phasor 는 Rv 를 **LORO 로** 고른다(격자에서 최댓값을 고르면 선택 편의 · R34 ④)
EV = {}
for inp in INPUTS:
    if ("dwt", inp) in DET:
        EV[f"dwt|{inp}"] = evaluate(("dwt", inp))
    per_rv = {rv: evaluate(("phasor", inp, rv)) for rv in RV_FRACS}
    sel = {}
    for rid in RIDS:
        cand = [rv for rv in RV_FRACS if rid in per_rv[rv]]
        if not cand:
            continue
        best = max(cand, key=lambda rv: np.mean(
            [min(per_rv[rv][o]["se"], per_rv[rv][o]["pp"]) for o in per_rv[rv] if o != rid]
            or [-np.inf]))
        sel[rid] = per_rv[best][rid] | {"rv": best}
    EV[f"phasor|{inp}"] = sel
run.log(f"  ({time.time()-T3_:.0f}초) 팔별 성적 (기권 문턱 LORO · 정답 전수 분모)")
TAB = {}
for a in ARMS:
    e = EV.get(a, {})
    if len(e) < 3:
        continue
    sm, slo, shi, n_ = boot_mean([e[r]["se"] for r in e], SEED0 + 11)
    pm, plo, phi_, _ = boot_mean([e[r]["pp"] for r in e], SEED0 + 12)
    er = np.abs(np.concatenate([e[r]["err"] for r in e]) if any(e[r]["err"] for r in e)
                else np.array([np.nan]))
    TAB[a] = dict(se=sm, se_lo=slo, se_hi=shi, pp=pm, pp_lo=plo, pp_hi=phi_, n=n_,
                  fire=float(np.mean([e[r]["fire"] for r in e])),
                  err_med=float(np.nanmedian(er)))
    run.log(f"    {a:<14} Se **{sm:.4f}** [{slo:.4f}, {shi:.4f}] · "
            f"PPV **{pm:.4f}** [{plo:.4f}, {phi_:.4f}] · 발화 {TAB[a]['fire']:.3f} · "
            f"|오차| {TAB[a]['err_med']:.1f}ms · n={n_}")

# ★★ 판정 팔도 **LORO 로** 고른다 — 표에서 최대를 고르면 그건 평가 지표로 고른 것이다
ARM_OK = [a for a in ARMS if a in TAB]
COMMON = sorted(set.intersection(*[set(EV[a]) for a in ARM_OK])) if ARM_OK else []
PICK = {}
for rid in COMMON:
    PICK[rid] = max(ARM_OK, key=lambda a: np.mean(
        [min(EV[a][o]["se"], EV[a][o]["pp"]) for o in COMMON if o != rid] or [-np.inf]))
se_s = [EV[PICK[r]][r]["se"] for r in COMMON]
pp_s = [EV[PICK[r]][r]["pp"] for r in COMMON]
sm, slo, shi, nsel = boot_mean(se_s, SEED0 + 13)
pm, plo, phi_, _ = boot_mean(pp_s, SEED0 + 14)
_cnt = {a: sum(1 for r in COMMON if PICK[r] == a) for a in ARM_OK}
_cnt = {k: v for k, v in _cnt.items() if v}
BEST_ARM = max(_cnt, key=_cnt.get) if _cnt else None
run.log(f"\n    ★★ **LORO 선택 팔**(판정 대상) 선택 분포 {_cnt}")
run.log(f"       Se **{sm:.4f}** [{slo:.4f}, {shi:.4f}] · PPV **{pm:.4f}** [{plo:.4f}, {phi_:.4f}]")
_omax = max((min(TAB[a]["se"], TAB[a]["pp"]) for a in ARM_OK), default=float("nan"))
run.log(f"       ▸ 표에서 최대를 고르면 min(Se,PPV) {_omax:.4f} · LORO 선택 {min(sm, pm):.4f}"
        " — 차이가 **선택 편의**다")
run.log(f"    벤치마크 (Saclova 2022 · phasor + 부정맥 결정 규칙) Se {BENCH_SE:.4f} / "
        f"PP {BENCH_PP:.4f} · 격차 Se {sm-BENCH_SE:+.4f} · PPV {pm-BENCH_PP:+.4f}")
run.log(f"    Q7-T 자체 검출기(argmax|잔차|) Se 0.4375 / PPV 0.3783 — 격차 {sm-0.4375:+.4f}")
if nsel < 3:
    g_("U1", "⛔ 측정 불가", "판정 가능한 레코드가 3 미만")
else:
    ok = (slo > SE_MIN) and (plo > PPV_MIN)
    bad = (shi < SE_MIN) or (phi_ < PPV_MIN)
    DIFF["U1"] = dict(arm="loro", pick=_cnt, se=sm, se_lo=slo, se_hi=shi,
                      pp=pm, pp_lo=plo, pp_hi=phi_, n=nsel)
    g_("U1", "✅ 지지" if ok else ("❌ 기각" if bad else "⚠️ 미결"),
       f"★★ Se {sm:.4f} [{slo:.4f}, {shi:.4f}] · PPV {pm:.4f} [{plo:.4f}, {phi_:.4f}] "
       f"· 문턱 {SE_MIN}/{PPV_MIN} · MDE Se {mde(slo, shi):.4f}")
CONFIG["U1"] = TAB; CONFIG["best_arm"] = BEST_ARM; CONFIG["pick"] = _cnt
run.save_json("config", CONFIG)

In [ ]:
# CELL 6 — 【U-D】 ★★ 주 관문 U2 (소거가 검출을 돕는가) · U3 재구성 비용 · U4 st vs abs
run.log("\n" + "=" * 100)
run.log("【U-D】 U2(주) — 소거가 **검출**을 돕는가 · U3 재구성 비용 · U4 st vs abs")
run.log("=" * 100)

def paired(det, a_in, b_in, seed):
    """같은 검출기·같은 레코드에서 입력만 바꾼 **짝지은 차**(a − b)."""
    ka, kb = f"{det}|{a_in}", f"{det}|{b_in}"
    if ka not in EV or kb not in EV:
        return None
    rs = sorted(set(EV[ka]) & set(EV[kb]))
    if len(rs) < 3:
        return None
    ds = [EV[ka][r]["se"] - EV[kb][r]["se"] for r in rs]
    dp = [EV[ka][r]["pp"] - EV[kb][r]["pp"] for r in rs]
    s = boot_mean(ds, seed); p = boot_mean(dp, seed + 1)
    return dict(n=len(rs), se=s[0], se_lo=s[1], se_hi=s[2],
                pp=p[0], pp_lo=p[1], pp_hi=p[2])

run.log("  U2 — 소거 vs **`none`(재구성만)** ★ 대조가 `raw` 가 아니라 `none` 인 게 핵심이다")
run.log("       재구성 인공물을 소거 탓으로 돌리지 않기 위해서(R34 ③)")
U2 = {}
for det in DETECTORS:
    for canc in ("abs", "st"):
        d_ = paired(det, canc, "none", SEED0 + 21)
        if d_ is None:
            continue
        U2[f"{det}|{canc}"] = d_
        run.log(f"    {det}|{canc:<4} vs none  ΔSe **{d_['se']:+.4f}** "
                f"[{d_['se_lo']:+.4f}, {d_['se_hi']:+.4f}] · ΔPPV **{d_['pp']:+.4f}** "
                f"[{d_['pp_lo']:+.4f}, {d_['pp_hi']:+.4f}] · n={d_['n']}")
if not U2:
    g_("U2", "⛔ 측정 불가", "짝지은 차를 못 냈다")
else:
    # ★ 여기서도 최량을 지표로 고르면 선택 편의 → LORO 선택
    K2 = sorted(U2)                                   # 예: "dwt|st"
    def _base(k):
        return k.split("|")[0] + "|none"
    R2 = sorted(set.intersection(*[set(EV[k]) & set(EV[_base(k)]) for k in K2]))
    def _d(k, r):
        return EV[k][r]["se"] - EV[_base(k)][r]["se"]
    sel = []
    for r in R2:                                      # ★ 팔 선택도 LORO(선택 편의 차단)
        pk = max(K2, key=lambda k: np.mean([_d(k, o) for o in R2 if o != r] or [-np.inf]))
        sel.append(_d(pk, r))
    m2, l2, h2, n2 = boot_mean(sel, SEED0 + 23)
    DIFF["U2"] = dict(arm="loro", mean=m2, lo=l2, hi=h2, n=n2, mde=float(mde(l2, h2)),
                      per_arm=U2)
    g_("U2", decide(l2, h2, 0.0, ">"),
       f"★★ **LORO 선택** ΔSe **{m2:+.4f}** [{l2:+.4f}, {h2:+.4f}] · n={n2} · "
       f"MDE {mde(l2, h2):.4f}")
    run.log("       ▸ > 0 이면 **소거 잔차에 공개 delineator 를 거는 게 원신호보다 낫다** —")
    run.log("         공개 방법은 전부 원신호에서만 한다. 이 조합이 이 퀘스트의 고유 기여다")

run.log("\n  U3 — 재구성 비용 (`none` vs `raw`)")
U3 = {}
for det in DETECTORS:
    d_ = paired(det, "none", "raw", SEED0 + 31)
    if d_ is None:
        continue
    U3[det] = d_
    run.log(f"    {det:<7} ΔSe {d_['se']:+.4f} [{d_['se_lo']:+.4f}, {d_['se_hi']:+.4f}] · "
            f"ΔPPV {d_['pp']:+.4f} [{d_['pp_lo']:+.4f}, {d_['pp_hi']:+.4f}] · n={d_['n']}")
run.log("    ▸ 음수면 **되붙이기 자체가 비용**이다 — U2 는 그 비용을 이미 뺀 값이다")

run.log("\n  U4 — 시공간 보정이 **검출에도** 이득인가 (`st` vs `abs`)")
U4 = {}
for det in DETECTORS:
    d_ = paired(det, "st", "abs", SEED0 + 41)
    if d_ is None:
        continue
    U4[det] = d_
    run.log(f"    {det:<7} ΔSe {d_['se']:+.4f} [{d_['se_lo']:+.4f}, {d_['se_hi']:+.4f}] · "
            f"ΔPPV {d_['pp']:+.4f} [{d_['pp_lo']:+.4f}, {d_['pp_hi']:+.4f}] · n={d_['n']}")
CONFIG["U2"] = U2; CONFIG["U3"] = U3; CONFIG["U4"] = U4
run.save_json("config", CONFIG)

In [ ]:
# CELL 7 — 【U-E】 U5 「P 부재」 재측정 · 그림 · 요약
run.log("\n" + "=" * 100)
run.log("【U-E】 U5 — 「P 부재」를 **작동하는 검출기로** 다시 잰다")
run.log("=" * 100)
run.log("  ▸ Q7-T 의 T3(AUROC 0.5190 ❌)은 **T1 이 실패한 검출기의 출력**을 써서")
run.log("    독립 증거가 아니었다. 검출기를 갈았으니 다시 묻는다")

def absence_auc(arm):
    """비트마다 「이 비트에 P 가 있나」 vs 검출 점수. 정답은 **주석 P 가 창 안에 있나**.
    반환은 **{rid: AUROC}** — 리스트로 돌려주면 LORO 선택에서 레코드가 어긋난다."""
    au = {}
    for rid in EV.get(arm, {}):
        d = BUT[rid]; fs = d["fs"]
        lo, hi = ms2s(P_LO_MS, fs), ms2s(P_HI_MS, fs)
        key = ("dwt", arm.split("|")[1]) if arm.startswith("dwt") else \
              ("phasor", arm.split("|")[1], EV[arm][rid].get("rv", RV_FRACS[0]))
        pos, sc = DET[key][rid]["pos"], DET[key][rid]["sc"]
        if not len(pos):
            continue
        y, s = [], []
        for p0 in d["r"]:
            a, b = p0 + lo, p0 + hi
            y.append(int(((d["p"] >= a) & (d["p"] < b)).any()))
            m = (pos >= a) & (pos < b)
            s.append(float(sc[m].max()) if m.any() else 0.0)
        y = np.asarray(y); s = np.asarray(s)
        if y.sum() >= 5 and (1 - y).sum() >= 5:              # R17
            au[rid] = float(roc_auc_score(y, s))
    return au

U5, AU = {}, {}
for a in ARMS:
    if a not in EV:
        continue
    AU[a] = absence_auc(a)
    if len(AU[a]) < 3:
        continue
    m_, lo_, hi_, n_ = boot_mean(list(AU[a].values()), SEED0 + 51)
    U5[a] = dict(auroc=m_, lo=lo_, hi=hi_, n=n_)
    run.log(f"    {a:<14} AUROC **{m_:.4f}** [{lo_:.4f}, {hi_:.4f}] · n={n_}")
if not U5:
    g_("U5", "⛔ 측정 불가", "층이 있는 레코드가 3 미만")
else:
    A5 = sorted(U5)
    R5 = sorted(set.intersection(*[set(AU[a]) for a in A5]))
    sel5 = []
    for r in R5:                                       # ★ 여기도 LORO 선택
        pk = max(A5, key=lambda a: np.mean([AU[a][o] for o in R5 if o != r] or [-np.inf]))
        sel5.append(AU[pk][r])
    m5, l5, h5, n5 = boot_mean(sel5, SEED0 + 52)
    if n5 < 3:
        g_("U5", "⛔ 측정 불가", "LORO 선택 표본이 3 미만")
    else:
        DIFF["U5"] = dict(arm="loro", auroc=m5, lo=l5, hi=h5, n=n5, mde=float(mde(l5, h5)))
        g_("U5", decide(l5, h5, U5_AUROC_MIN, ">"),
           f"★ **LORO 선택** AUROC **{m5:.4f}** [{l5:.4f}, {h5:.4f}] · 문턱 {U5_AUROC_MIN}"
           f" · n={n5} (Q7-T T3 은 0.5190)")

# ── 그림 ⚠️ Colab 기본 matplotlib 에 **한글이 없어** 축·범례는 ASCII 로만 쓴다.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.6))
aa = [a for a in ARMS if a in TAB]
ax[0].scatter([TAB[a]["pp"] for a in aa], [TAB[a]["se"] for a in aa], s=48)
for a in aa:
    ax[0].annotate(a, (TAB[a]["pp"], TAB[a]["se"]), fontsize=6.5,
                   xytext=(3, 3), textcoords="offset points")
ax[0].scatter([BENCH_PP], [BENCH_SE], marker="*", s=200, color="tab:red")
ax[0].annotate("Saclova 2022", (BENCH_PP, BENCH_SE), fontsize=7, color="tab:red",
               xytext=(4, -10), textcoords="offset points")
ax[0].scatter([0.3783], [0.4375], marker="x", s=70, color="tab:gray")
ax[0].annotate("Q7-T own", (0.3783, 0.4375), fontsize=7, color="tab:gray",
               xytext=(4, -10), textcoords="offset points")
ax[0].axhline(SE_MIN, ls=":", lw=.9, color="k"); ax[0].axvline(PPV_MIN, ls=":", lw=.9, color="k")
ax[0].set_xlabel("PPV"); ax[0].set_ylabel("Se"); ax[0].set_xlim(0, 1.02); ax[0].set_ylim(0, 1.02)
ax[0].grid(alpha=.3)

k2 = sorted(U2)
ax[1].errorbar([U2[k]["se"] for k in k2], np.arange(len(k2)),
               xerr=[[U2[k]["se"] - U2[k]["se_lo"] for k in k2],
                     [U2[k]["se_hi"] - U2[k]["se"] for k in k2]],
               fmt="o", capsize=4, color="tab:blue", label="dSe")
ax[1].axvline(0, color="k", lw=.9)
ax[1].set_yticks(range(len(k2))); ax[1].set_yticklabels(k2, fontsize=7.5)
ax[1].set_xlabel("dSe : cancellation minus reconstruction-only")
ax[1].grid(alpha=.3, axis="x"); ax[1].legend(fontsize=7)

a5 = [a for a in ARMS if a in U5]
ax[2].bar(np.arange(len(a5)), [U5[a]["auroc"] for a in a5])
ax[2].axhline(U5_AUROC_MIN, ls=":", lw=1.0, color="k")
ax[2].axhline(0.5, ls="--", lw=.8, color="tab:gray")
ax[2].set_xticks(range(len(a5)))
ax[2].set_xticklabels(a5, fontsize=6.5, rotation=30, ha="right")
ax[2].set_ylabel("P-absence AUROC"); ax[2].set_ylim(0, 1.0); ax[2].grid(alpha=.3, axis="y")
fig.tight_layout()
PNG = run.save_fig("q7u_public_delineator", fig)
plt.close(fig); display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("요약")
run.log("=" * 100)
ok_ = lambda k: VERD.get(k, "").startswith("✅")
no_ = lambda k: VERD.get(k, "").startswith("❌")
un_ = lambda k: VERD.get(k, "").startswith("⛔")     # R29 ②
for g in ("U1", "U2", "U5"):
    run.log(f"  {g:<4}{VERD.get(g, '(미실행)')}")
run.log("")
if any(un_(g) for g in ("U1", "U2")):
    run.log("  ⛔ 측정 불가가 있다 — 어떤 결론 분기도 타지 않는다(R29 ②)")
elif ok_("U1") and ok_("U2"):
    run.log("  ★★ **자가 서고, 소거가 검출을 돕는다.**")
    run.log("     공개 방법은 전부 **원신호**에서만 P 를 찾는다 — 소거 잔차에 거는 건")
    run.log("     아무도 안 한 조합이고, 이 퀘스트의 **고유 기여**다.")
    run.log("     → **Q7-S′** 로 간다 — Q7-S 설계에서 `feats_for()` 만 **P 정렬 특징**으로")
elif ok_("U1") and no_("U2"):
    run.log("  ⚠️ **자는 섰는데 소거는 검출에 안 보탠다.** T2(순위 +0.2825)와 U2 가 갈린다 —")
    run.log("     소거는 P 를 **드러내지만** 검출 성능으론 안 옮겨진다.")
    run.log("     → 소거는 **특징 생성에만** 쓰고 검출은 원신호에서 한다")
elif no_("U1"):
    run.log("  ⛔ **공개 방법으로도 자가 안 선다.** 이 규모에서 P 검출은 우리 문제가 아니다.")
    run.log("     → **형태 갈래를 닫거나** 학습형 검출기로 간다(외부 정답 성격을 잃는 대가)")
else:
    run.log("  ⚠️ 미결 — MDE 와 비교해 「효과 없음」인지 「측정 한계」인지 먼저 가른다(R33 ①)")
    for g in ("U1", "U2", "U5"):
        d_ = DIFF.get(g)
        if d_ and "mde" in d_ and "mean" in d_:
            nn = need_n(d_.get("n", 0), d_["lo"], d_["hi"], d_["mean"], 0.05)
            if nn is not None and np.isfinite(nn):
                run.log(f"     {g} 필요 레코드 ≈ {nn:.0f} (현재 {d_.get('n')} · BUT PDB 는 50)")
if ok_("U5"):
    run.log("  ★ **「P 부재」가 특징이 된다** — Q7-T 의 T3(0.5190)은 실패한 검출기 탓이었다")

run.log("\n  사전등록 **종결 조건** (R34 ⑤)")
run.log("    ① U1 이 ❌ 면 **형태 갈래를 닫는다** — 공개 방법으로도 못 세운 자를 우리가")
run.log("       더 만들지 않는다. 학습형으로 갈지는 별도 결정이다")
run.log("    ② U2 가 ❌ 면 소거는 **검출 경로에서 내린다**(특징 생성에는 남긴다 — T2 ✅)")
run.log("    ③ U1·U2 가 둘 다 미결이고 필요 레코드가 BUT PDB 50개를 넘으면 같은 자료로")
run.log("       **더 돌지 않는다** — LUDB·QT DB 로 정답을 늘린다")

run.log("\n  ▸ 이 런도 **SVEB 질문에 답하지 않는다** — 「P 를 볼 수 있는가 · 소거가 돕는가」까지")
run.log("  ▸ `phasor` 는 Martinez 2010 의 **P 정점만 잡는 축소판**이다")
run.log("  ▸ `st` 는 Stridh-Sornmo 의 **2유도 축소판**이다")

run.finish({
    "exp_id": "quest46_q7u_public_delineator",
    "metric": "butpdb_delta_se_cancel_vs_recon",
    "value": float(DIFF.get("U2", {}).get("mean", float("nan"))),
    "passed": bool(ok_("U1") and ok_("U2")),
    "summary": ("공개 delineator(NeuroKit2 DWT · phasor transform 축소판)를 자로 빌려, "
                "**QRST 소거 잔차에 걸면 원신호보다 나은가**를 물었다. 주 관문은 U2."),
    "verdicts": VERD, "diffs": DIFF, "rule_check": RULE_CHECK,
    "U1": CONFIG.get("U1", {}), "U2": CONFIG.get("U2", {}), "U3": CONFIG.get("U3", {}),
    "U4": CONFIG.get("U4", {}), "U5": U5, "but": CONFIG.get("but", {}),
    "input_rms": CONFIG.get("input_rms", {}), "best_arm": BEST_ARM,
    "pick": CONFIG.get("pick", {}), "benchmark": CONFIG.get("benchmark", {}), "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `ingest_run.py --quest ailab-2026-0046 --step public-delineator`")